# Bratteli diagram of the walled Brauer algebras (Figs. 1-3)

This notebook draws the Bratteli diagram of the family $B_{0,0} \to B_{1,0} \to \dots \to B_{p,0} \to B_{p,1} \to \dots \to B_{p,q}$ of walled Brauer algebras and marks the part that is removed for the matrix algebras $A^d_{p,q}$.

* **Vertices** are triples `(lambda_l, lambda_r, level)`, where $(\lambda^l, \lambda^r)$ is a bipartition of eq. (84) at level $0, \dots, p+q$. The level tells apart equal bipartitions on different levels, e.g. $(\emptyset, \emptyset)$ at levels 0 and 4 when $p = q = 2$.
* **Edges** follow Sect. 5.1 (`wba_covers`): up to level $p$ a cell is added to $\lambda^l$; after level $p$ either a cell is added to $\lambda^r$ or a cell is removed from $\lambda^l$.
* **Edge labels** are the contents $c_{T[k-1] \to T[k]}$ of Sect. 5.3 with $\delta = d$ (`compute_content`).
* **Red edges** touch a vertex with $\mathrm{len}(\lambda^l) + \mathrm{len}(\lambda^r) > d$, i.e. they are removed by the truncation of Sect. 5.4 (`color_edge`). The black edges form the Bratteli diagram of $A^d_{0,0} \to \dots \to A^d_{p,q}$.

Parameters for the figures of the paper (`p`, `q` in the third code cell, `d` in the fifth):

| Figure | `p` | `q` | `d` |
|---|---|---|---|
| Fig. 1 (Young's lattice) | 4 | 0 | 2 or 3 |
| Fig. 2 ($B^\delta_{2,2}$) | 2 | 2 | $\ge 4$ (no red edges) |
| Fig. 3 ($A^d_{2,2}$) | 2 | 2 | 2 or 3 |

The notebook is set up for Fig. 3 with $d = 2$. It produces drafts: the published figures were finished by hand (column headers, vertex labels $(\lambda^l, \lambda^r)$ without the level, `d=2`/`d=3` tags instead of red edges, the highlighted path in Fig. 3, no edge labels).

**Requirements.** Building the diagram (the first five code cells) needs SageMath only. The drawings also need `dot2tex` and `graphviz`: without them Sage falls back to the `tkz_graph` format, which ignores the layered layout (`rankdir`), the edge colours and the edge labels. `view(H)` and `TikzPicture(...).pdf()` need a LaTeX installation with `tikz`, `tkz-graph` and `tkz-berge`.

In [ ]:
latex.extra_preamble('\\usepackage{tikz}\n\\usepackage{tkz-graph}\n\\usepackage{tkz-berge}\n\\usetikzlibrary{arrows,shapes}')
latex.add_to_preamble("\\renewcommand{\\emptyset}{\\varnothing}")

In [ ]:
def wba_covers(x,y):
    if x[2]+1==y[2]:
        if x[0].size()==x[2] and y[0].size()==y[2]:
            return y[0].contains(x[0])
        elif x[0].size()<=x[2] and y[0].size()<y[2]:
            if y[0].size()==x[0].size():
                return y[1].contains(x[1]) and x[0].contains(y[0])
            elif y[1].size()==x[1].size():
                return x[0].contains(y[0]) and x[1].contains(y[1])         
    return False

def color_edge(edge,d):
    if (edge[0][0].length()+edge[0][1].length() > d) or (edge[1][0].length()+edge[1][1].length() > d):
        return 'red'
    else:
        return 'black'
    
def compute_content(edge,p,z=var('delta') ):
    inp = edge[0]
    out = edge[1]
    if out[2] <= p:
        cell = Set(out[0].cells()).symmetric_difference(Set(inp[0].cells()))[0]
        content = cell[1]-cell[0]
    else:
        length = len(Set(out[0].cells()).symmetric_difference(Set(inp[0].cells())))
        if length==1:
            cell = Set(out[0].cells()).symmetric_difference(Set(inp[0].cells()))[0]
            content = cell[0]-cell[1]
        else:
            cell = Set(out[1].cells()).symmetric_difference(Set(inp[1].cells()))[0]
            content = cell[1]-cell[0]+z
    return (edge[0],edge[1],content)

In [ ]:
p = 2  # Fig. 3 of the paper (see the table above)
q = 2
elements_up_to_p = [(b,Partition([]),n) for n in range(p+1) for b in Partitions(n)]
elements_after_p = [(b1,b2,n+p) for n in range(1,q+1) for k in range(min(p,n),-1,-1) for b1 in Partitions(p-k) for b2 in Partitions(n-k)]
#print(elements_up_to_p,'\n' )
#print(elements_after_p)

In [ ]:
elements = elements_up_to_p + elements_after_p
Y = Poset((elements,wba_covers), facade=True)
H = Y.hasse_diagram()

In [ ]:
d = 2  # Fig. 3 uses d = 2 or 3 (see the table above)
for e in H.edges():
    H.delete_edge(e)
    H.add_edge(compute_content(e,p,d))
    
edge_colors={}
for e in H.edges():
    edge_colors[e] = color_edge(e,d)
     
# vertex_labels={}
# for v in H.vertices():
#     vertex_labels[v] = '$'+str(latex(v[0]))+', '+str(latex(v[1]))+', %d'%v[2]+'$'
#     vertex_labels[v] = latex(v)
#     print(vertex_labels[v])
# H.relabel(vertex_labels)

In [ ]:
opts = H.latex_options()
opts.set_options(edge_colors=edge_colors,edge_labels=True,tkz_style = 'Art',rankdir='down')
#opts.set_options(tkz_style = 'Art',rankdir='down',)
view(H)

In [ ]:
latex(H)

In [ ]:
print(latex.extra_preamble())

### another method

In [ ]:
from sage.misc.latex_standalone import TikzPicture
path_to_file = TikzPicture.from_poset(Y,rankdir='right').pdf()  
path_to_file_h = TikzPicture.from_graph(H,rankdir='right').pdf() 